# 基于transformer的中英文翻译   
## 一、简介   
手写实现transformer模型的各个组件，包括多头注意力，前馈神经网络，位置编码等。    
步骤    
1. 数据预处理   
2. 数据加载     
3. 模型构建   
4. 模型训练与验证   
5. 推理   

## 二、 环境   
python 3.12   
pytorch   
Pandas   
jieba   
torchtext   
Scikit-learn   
注意：torch与torchtext有版本兼容问题，torchtext好像已经停止更新，建议使用低版本的torch   
```bash
pip install torch==2.3.0 torchvision==0.18.0
```
## 三、数据集   
中英文句子对，英文与中文句子以制表符分隔   
[dataset](http://www.manythings.org/anki/cmn-eng.zip)   
## 四、实现   
### 1. 数据预处理   
从数据集提取英文和中文句子存入单独的文件，转换输入格式   
任务：   
1. 读取文件    
2. 提取句子：分割并提取句子   
3. 保存数据：保存为两个单独文件，一个保存英文句子，一个保存中文句子   


In [6]:
import pandas as pd


data = []
# 加载文件
file_path = 'data/cmn.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        #按制表符来分割 
        parts = line.strip().split('\t')
        # 去除缺失的数据   
        if len(parts) >= 2:
            english = parts[0].strip()
            chinese = parts[1].strip()
            data.append([english,chinese])

# 使用DataFrame,column指定列名
df = pd.DataFrame(data,columns=['English','Chinese'])
# 测试df
print(df.head(10))

# 保存到文件  index会有整数行索引 header会有列名
df['English'].to_csv('data/English.txt',index=False,header=False)
df['Chinese'].to_csv('data/Chinese.txt',index=False,header=False)

  English Chinese
0     Hi.      嗨。
1     Hi.     你好。
2    Run.   你用跑的。
3    Who?      誰？
4   Fire!      火！
5   Fire!     火啊！
6   Stay.     待著。
7   Stay.     且慢。
8   Stop!     住手！
9   Wait!     等等！


### 2. 数据加载   
1. 分词   
2. 数据处理-padding使得句子的长度一致(输入数据处理核心)   
3. 迭代器加载数据   

In [27]:
import torch
import jieba
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchtext
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from sklearn.model_selection import train_test_split


torchtext.disable_torchtext_deprecation_warning()
# 分词器
# 英文使用 torchtext自带分词器
tokenizer_en = get_tokenizer('basic_english')

# 中文使用jieba  cut返回迭代器 lcut返回列表
def tokenizer_zh(text):
    return jieba.lcut(text)

# 构建生成器   输入句子和分词器(ch/en)
def tokens(sentences, tokenizer):
    return [tokenizer(sentence) for sentence in sentences]

#
def build_vocabs(en_sentences,zh_sentences):
    # 定义特殊符号  
    # 训练的基础 未知词  填充 开始 结束
    # transformer是固定长度的任务，如果不一样需要填充
    # 开始结束标记生成任务的开始结束
    # 未知词共享一个向量标记未知
    specials = ['<unk>','<pad>','<bos>','<eos>']

    # 将四个特殊词加到词汇表头部,同时给出每一个索引
    # 英语词汇表
    en_vocab = build_vocab_from_iterator(
        tokens(en_sentences,tokenizer_en),
        specials=specials,
    )

    # 中文词汇表
    zh_vocab = build_vocab_from_iterator(
        tokens(zh_sentences,tokenizer_zh),
        specials=specials,
    )

    # 设置默认索引
    en_vocab.set_default_index(en_vocab['<unk>'])
    zh_vocab.set_default_index(zh_vocab['<unk>'])

    return en_vocab, zh_vocab

# 句子转换tensor  将数据转换成transformer可训练数据
def text_transform(sentence,tokenizer,vocab):
    # 分词
    ts = tokenizer(sentence)
    ids = [vocab['<bos>']] + [vocab[t] for t in ts] + [vocab['<eos>']]
    return torch.tensor(ids)


In [28]:
# 自定义数据导入
class TranslationDataset(Dataset):
    # 英文原始数据  中文原始数据  加入特殊符号的中英文数据集
    def __init__(self,en_sentenses,ch_sentenses,en_vocab,zh_vocab):
        self.data = [] 

        # 处理
        for en, zh in zip(en_sentenses, ch_sentenses):
            en_tensor = text_transform(en, tokenizer_en, en_vocab)
            zh_tensor = text_transform(zh, tokenizer_zh, zh_vocab)
            #加入data
            self.data.append((en_tensor,zh_tensor))

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        return self.data[index]

# 处理batch 将句子对齐，不足的句子补充<pad>
def collate_fn(batch):
    en_batch,zh_batch = [],[]
    for en_item,ch_item in batch:
        en_batch.append(en_item)
        zh_batch.append(ch_item)

    # 填充  使用<pad>索引填入
    en_batch = pad_sequence(en_batch,padding_value=1,batch_first=True)
    zh_batch = pad_sequence(zh_batch,padding_value=1,batch_first=True)

    return en_batch,zh_batch


In [29]:
# 测试
en_data = ["I love machine learning.", "The weather is nice today."]
zh_data = ["我喜欢机器学习。", "今天天气不错。"]
    
# 构建词汇表
en_vocab, zh_vocab = build_vocabs(en_data, zh_data)

print(en_vocab.get_itos())
print(zh_vocab.get_itos())

print(en_vocab['<bos>'])

for en, zh in zip(en_data, zh_data):
    en_tensor = text_transform(en, tokenizer_en, en_vocab)
    zh_tensor = text_transform(zh, tokenizer_zh, zh_vocab)
    print(en_tensor)
    print(zh_tensor)


dataset = TranslationDataset(en_data, zh_data, en_vocab, zh_vocab)
    
# 创建 DataLoader
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

for idx,(en,zh) in enumerate(dataloader):
    print("英文形状:",en.shape)
    print("中文形状:",zh.shape)
    print(en)
    print(zh)

['<unk>', '<pad>', '<bos>', '<eos>', '.', 'i', 'is', 'learning', 'love', 'machine', 'nice', 'the', 'today', 'weather']
['<unk>', '<pad>', '<bos>', '<eos>', '。', '不错', '今天天气', '喜欢', '学习', '我', '机器']
2
tensor([2, 5, 8, 9, 7, 4, 3])
tensor([ 2,  9,  7, 10,  8,  4,  3])
tensor([ 2, 11, 13,  6, 10, 12,  4,  3])
tensor([2, 6, 5, 4, 3])
英文形状: torch.Size([2, 8])
中文形状: torch.Size([2, 7])
tensor([[ 2, 11, 13,  6, 10, 12,  4,  3],
        [ 2,  5,  8,  9,  7,  4,  3,  1]])
tensor([[ 2,  6,  5,  4,  3,  1,  1],
        [ 2,  9,  7, 10,  8,  4,  3]])


* 构建数据集   

In [ ]:
# 文件中加载句子
with open('data/English.txt','r',encoding='utf-8') as f:
    english_sentences = [line.strip() for line in f]

with open('data/Chinese.txt','r',encoding='utf-8') as f:
    chinese_sentences = [line.strip() for line in f]

#构建词汇表
en_vocab, zh_vocab = build_vocabs(english_sentences, chinese_sentences)

print(f'英文词汇表大小:{len(en_vocab)}')
print(f'中文文词汇表大小:{len(zh_vocab)}')

dataset = TranslationDataset(english_sentences,chinese_sentences,en_vocab, zh_vocab)

# 划分数据集
train_data,val_data = train_test_split(dataset,test_size=0.1)

# 创建加载器
batch_size = 32
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_data, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)


# 这里观察到每一个维度不一样，每一个batch独立padding
for idx,(en,zh) in enumerate(train_dataloader):
    print("英文训练加载器形状:",en.shape)
    print("中文训练加载器形状:",zh.shape)

for idx,(en,zh) in enumerate(val_dataloader):
    print("英文验证加载器形状:",en.shape)
    print("中文验证加载器形状:",zh.shape)



英文词汇表大小:7479
中文文词汇表大小:17200
英文训练加载器形状: torch.Size([32, 14])
中文训练加载器形状: torch.Size([32, 16])
英文训练加载器形状: torch.Size([32, 20])
中文训练加载器形状: torch.Size([32, 15])
英文训练加载器形状: torch.Size([32, 18])
中文训练加载器形状: torch.Size([32, 18])
英文训练加载器形状: torch.Size([32, 20])
中文训练加载器形状: torch.Size([32, 17])
英文训练加载器形状: torch.Size([32, 19])
中文训练加载器形状: torch.Size([32, 16])
英文训练加载器形状: torch.Size([32, 17])
中文训练加载器形状: torch.Size([32, 21])
英文训练加载器形状: torch.Size([32, 14])
中文训练加载器形状: torch.Size([32, 12])
英文训练加载器形状: torch.Size([32, 13])
中文训练加载器形状: torch.Size([32, 12])
英文训练加载器形状: torch.Size([32, 18])
中文训练加载器形状: torch.Size([32, 17])
英文训练加载器形状: torch.Size([32, 16])
中文训练加载器形状: torch.Size([32, 17])
英文训练加载器形状: torch.Size([32, 19])
中文训练加载器形状: torch.Size([32, 14])
英文训练加载器形状: torch.Size([32, 21])
中文训练加载器形状: torch.Size([32, 15])
英文训练加载器形状: torch.Size([32, 16])
中文训练加载器形状: torch.Size([32, 15])
英文训练加载器形状: torch.Size([32, 16])
中文训练加载器形状: torch.Size([32, 13])
英文训练加载器形状: torch.Size([32, 15])
中文训练加载器形状: torch.Size([32, 13])
英文训练加载器形状: t

### 3. transformer架构
#### 3.1 多头注意力   
* 自注意力   
```math
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
```

| 符号 | 含义 | 维度 |
|------|------|------|
| Q | 查询矩阵 (Query)：表示当前时刻寻找的信息 | (n, dₖ) |
| K | 键矩阵 (Key)：表示输入序列中各信息点的标签 | (n, dₖ) |
| V | 值矩阵 (Value)：表示输入序列中各信息点的实际内容 | (n, dᵥ) |
| dₖ | 键向量的维度（用于缩放因子）,dₖ是方差 | 标量 |
| n | 序列长度 | 标量 |

计算：   
1. 线性运算   
```math
Q = XW^Q, \quad K = XW^K, \quad V = XW^V
```
2. 注意力得分   
通过计算 Q 和 K 的点积来衡量词与词之间的相关性：   
```math
\text{Scores} = QK^T
```
3. 缩放归一    
除以维度开方进行缩放，并通过 Softmax 得到权重系数 $\alpha$：   
维度是方差，维度开方是标准归一化，降维同时是训练更加收敛   
```math
\alpha = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)
```
4. 加权求和   
```math
\text{Output} = \alpha V
```
